# Sivilstand og familieutvikling — Folkeregister

Illustrerer endringer i sivilstandsmønstre og familiestørrelser på tvers av aldersgrupper og år.

**Kilde:** `folkeregister.marital_status_trends`, `folkeregister.household_structure` (Gold-lag)  
**Relatert issue:** US E-4 / #96

In [ ]:
import os
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("INSTALL delta;  LOAD delta;")

_raw = os.environ.get("MINIO_ENDPOINT", "http://minio.slettix-analytics.svc.cluster.local:9000")
_endpoint = _raw.replace("http://", "").replace("https://", "")

con.execute(f"""
    CREATE OR REPLACE SECRET minio (
        TYPE      S3,
        KEY_ID    '{os.environ.get("MINIO_ACCESS_KEY", "admin")}',
        SECRET    '{os.environ.get("MINIO_SECRET_KEY", "changeme")}',
        ENDPOINT  '{_endpoint}',
        URL_STYLE 'path',
        USE_SSL   false
    );
""")

MARITAL_PATH    = "s3://gold/folkeregister/marital_status_trends"
HOUSEHOLD_PATH  = "s3://gold/folkeregister/household_structure"
print("DuckDB klar.", duckdb.__version__)

In [ ]:
df_mar  = con.execute(f"SELECT * FROM delta_scan('{MARITAL_PATH}')").df()
df_hous = con.execute(f"SELECT * FROM delta_scan('{HOUSEHOLD_PATH}')").df()
print(f"Sivilstandsrader: {len(df_mar):,} | Husstandsrader: {len(df_hous):,}")
print("Sivilstatus-verdier:", sorted(df_mar["marital_status"].unique()))
print("Husstandstyper:",       sorted(df_hous["household_type"].unique()))

## 1. Stacked area — sivilstandsfordeling over tid (alle aldersgrupper)

In [ ]:
# Aggreger over alle aldersgrupper
pivot = (
    df_mar.groupby(["reference_year", "marital_status"])["count"]
    .sum()
    .unstack(fill_value=0)
    .sort_index()
)
# Konverter til prosent
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

STATUS_COLORS = {
    "UGIFT":         "#3498db",
    "GIFT":          "#2ecc71",
    "SKILT":         "#e67e22",
    "ENKE_ENKEMANN": "#95a5a6",
}

fig, ax = plt.subplots(figsize=(11, 5))
years  = pivot_pct.index
bottom = np.zeros(len(years))

for status in pivot_pct.columns:
    vals = pivot_pct[status].values
    ax.fill_between(years, bottom, bottom + vals,
                    alpha=0.85,
                    color=STATUS_COLORS.get(status, "#bdc3c7"),
                    label=status.replace("_", "/"))
    bottom += vals

ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.set_ylim(0, 100)
ax.set_xlabel("År")
ax.set_ylabel("Andel av befolkning")
ax.set_title("Sivilstandsfordeling over tid", fontweight="bold")
ax.legend(loc="upper right", fontsize=9)
plt.tight_layout()
plt.show()

## 2. Sivilstandsfordeling per aldersgruppe — siste tilgjengelige år

In [ ]:
last_year = df_mar["reference_year"].max()
df_last = df_mar[df_mar["reference_year"] == last_year]

pivot_age = (
    df_last.groupby(["age_group", "marital_status"])["count"]
    .sum()
    .unstack(fill_value=0)
)
# Sorter aldersgrupper
age_order = sorted(pivot_age.index, key=lambda x: int(x.split("-")[0].replace("+", "")) if any(c.isdigit() for c in x) else 999)
pivot_age = pivot_age.reindex(age_order)
pivot_age_pct = pivot_age.div(pivot_age.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(11, 5))
bottom = np.zeros(len(pivot_age_pct))
for status in pivot_age_pct.columns:
    ax.bar(pivot_age_pct.index, pivot_age_pct[status],
           bottom=bottom, label=status.replace("_", "/"),
           color=STATUS_COLORS.get(status, "#bdc3c7"))
    bottom += pivot_age_pct[status].values

ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.set_xlabel("Aldersgruppe")
ax.set_ylabel("Andel")
ax.set_title(f"Sivilstandsfordeling per aldersgruppe ({last_year})", fontweight="bold")
plt.xticks(rotation=30, ha="right")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 3. Husstandstype og størrelseskategori

In [ ]:
pivot_h = (
    df_hous.pivot_table(index="household_type", columns="size_category",
                        values="count", aggfunc="sum")
    .fillna(0)
)

fig, ax = plt.subplots(figsize=(10, 5))
pivot_h.plot(kind="bar", ax=ax, colormap="Set2", edgecolor="white")
ax.set_xlabel("Husstandstype")
ax.set_ylabel("Antall")
ax.set_title("Husstandstype × størrelseskategori", fontweight="bold")
plt.xticks(rotation=20, ha="right")
ax.legend(title="Størrelse", fontsize=8)
plt.tight_layout()
plt.show()

## 4. Prosentandel husstandstyper (donut-chart)

In [ ]:
hous_tot = df_hous.groupby("household_type")["count"].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
wedges, texts, autotexts = ax.pie(
    hous_tot.values,
    labels=hous_tot.index,
    autopct="%1.1f%%",
    startangle=90,
    pctdistance=0.82,
    wedgeprops={"width": 0.5},
    colors=plt.get_cmap("Set2").colors[:len(hous_tot)],
)
for t in autotexts:
    t.set_fontsize(9)
ax.set_title("Fordeling av husstandstyper", fontweight="bold", pad=20)
plt.tight_layout()
plt.show()

print("\nTotal fordeling:")
print(df_hous.groupby("household_type")[["count", "pct_of_total"]].sum().sort_values("count", ascending=False).to_string())